# 07 · Explore CellBridge predictions by prediction-set size

Browse saved **CellBridge branch-graph** predictions against cell morphology; no inference or GPU is required. Notebook `06_conformal_prediction.ipynb` reports quantitative conformal results.

Run the cells in order with the lab's **hest** kernel. The HEST map defaults to **TENX201**, **512 px** patches, ranked by **mean saved prediction-set size**.


## Reading the plots

The four panels show **H&E**, **reference labels**, **highest-scoring predictions**, and **saved prediction-set size per cell**. Class colours remain sample-specific; macrophages are bright orange. Dots have black edges. Cells without saved test predictions are omitted from overlays.

**Set size** is the number of labels in the saved final conformal set. Branch-graph sets already include ontology closure. **Mean set size** averages these counts over all evaluated cells in the patch, including incorrect predictions and empty sets (size 0). It does not require ground-truth labels. One cell has a set size; a patch has a mean set size.

The scale runs **green (singleton) to red (all classes)**; grey is an empty set. The range is fixed to the section's class count. This measures set breadth, not a calibrated confidence probability. Coverage and top-1 accuracy are reported separately using the reference labels.


In [ ]:
import importlib
import sys
from pathlib import Path
from IPython.display import display

# Find the checkout when launched from the repository, notebooks/, or workspace.
REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / "projects/cellbridge")
             if (p / "src/cellbridge/config.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Launch Jupyter from the cellbridge checkout.")
sys.path.insert(0, str(REPO / "src"))
from cellbridge.qualitative import data, figures, views

# Rerunning setup picks up edits even when the kernel is already running.
for module in (data, figures, views):
    importlib.reload(module)

## Available datasets

Menus show saved predictions only. Fields are ordered by evaluated-cell count.

In [ ]:
catalog = views.show_catalog()

## Saving your review

**Save displayed view** exports the rendered figure as PDF, cell or patch tables, and provenance to:

```text
<workspace>/outputs/visualizations/res_figures/qualitative_conformal/<username>/<dataset>/<timestamp>/
```

Each save creates a new folder. After changing controls, render again before saving.

## Explore

1. **Load dataset**, then choose a **Field** and **Patch**.
2. Choose **Patch** or **Cell**. **Find cell** searches IDs and original annotations.
3. Click **Show selection** to inspect the four panels and the mean saved set size.

Optional **Smooth set-size map** shows a local mean over evaluated cells, including empty sets. The printed patch mean always uses unsmoothed per-cell sizes.


In [ ]:
explorer = views.Explorer(catalog, metric="set_size")
explorer.dataset_widget.value = next(c for c in catalog.index if catalog.loc[c, "section"] == "TENX201")
explorer.show()


## HEST tissue map · find examples

1. Choose **Section**, **Patch (px)** and **Min cells**, then click **Show tissue map**.
2. The default statistic is **Mean prediction-set size**. Patches are listed largest mean first; hover the map to see values and coordinates.
3. Choose a patch and click **Open selected patch** for H&E, reference, prediction, and per-cell set sizes.
4. **Save displayed view** saves PDFs, cell/patch CSV tables, and provenance in a new folder.

All evaluated cells contribute to the mean, even if their reference is excluded. Empty sets count as zero; missing test predictions are omitted. Check **Original-set coverage** (coverage of the saved set) and **Top-1 accuracy** alongside set size. Increase **Min cells** for more stable averages. Class counts differ across sections, so use the numerical values and class count when comparing them.

The optional table below lets you sort patch means in either direction without rescoring the section.


In [ ]:
tissue_map = views.TissueMap(catalog, metric="set_size")
tissue_map.section_widget.value = next(c for c in catalog.index if catalog.loc[c, "section"] == "TENX201")
tissue_map.tissue_size_widget.value = 512
tissue_map.show()


In [ ]:
# Rerun after changing the section, patch size, or minimum cell count.
# Set ascending=True to find compact sets; False to find broad sets.
if tissue_map.map is not None:
    display(tissue_map.map["table"].sort_values("mean_set_size", ascending=False).head(30))


## TERRA transcriptomic view

Select a HEST patch above, then choose an embedding and context and click **Fetch TERRA embeddings**. Colour defaults to **Prediction-set size**. Neighbourhood agreement is grouped by **set size**, including zero for empty sets.

HEST reference labels also come from transcript counts, so TERRA is supporting context rather than independent validation. Neighbour agreement depends on the sampled context; UMAP is refitted on each click.


In [ ]:
terra = views.TerraView(tissue_map).show()

## Review notes

Record patch coordinates, mean saved set size, evaluated-cell count, coverage and top-1 accuracy. Include failures as well as successful examples. A correct top-1 prediction can still have a broad conformal set.

Selected examples do not estimate whole-dataset performance. The displayed crop may differ from the model input crop.
